## **The purpose of this notebook is to construct the necessary data structures and conduct the LEiDA analysis.**

In [ ]:
# Import basic libraries for file operations, data handling, and progress bars
import os
import pandas as pd
import numpy as np
from tqdm import tqdm


In [ ]:
# Filter BOLD data to keep only ROIs belonging to the selected RSN(s)
def choose_rsn(data, rsn = None):
    bold = data.copy()
    if rsn==None:
        return bold
    else:
        rois = []
        for icn in rsn:
            rois += atlas[atlas['ICN'] == icn]['Name'].tolist()
        return bold.loc[rois,:]

# Get coordinates and labels for ROIs in the selected RSN(s)
def chosen_rois(rsn = None):
    if rsn == None:
        rois_coordinates = pd.DataFrame(atlas.loc[:,['R','A','S']])
        rois_labels = pd.DataFrame(atlas['Name'])
    else:
        rois_coordinates = []
        rois_labels = []
        for icn in rsn:
            rois_labels += atlas[atlas['ICN'] == icn]['Name'].tolist()
            rois_coordinates.append(atlas[atlas['ICN'] == icn].loc[:,['R','A','S']])
        rois_coordinates = pd.concat(rois_coordinates)
        rois_labels = pd.DataFrame(rois_labels,columns = ['Name'])

    return rois_coordinates,rois_labels


In [ ]:
# Load responder status and define subject/session lists
# Subject sub-010 was excluded due to an average head motion exceeding 0.5 mm.
responder = pd.read_csv(f"/data/dy/TIS_MDD/responder.csv", index_col = 0).drop("sub-010", axis = 0)
sub_ls = responder.index.tolist()
ses_ls = ['baseline','5th_201','5th_301','5th_after','4weeks']


In [ ]:
# Set atlas resolution, target networks, and output path for LEiDA results
schaefer = 100
chosen_network = ['Limbic',"Default"] 
leida_path = f"/data/dy/TIS_MDD/LEiDA/LIM_DMN/schaefer-{schaefer}/"


In [ ]:
# Load Schaefer atlas with RSN labels and MNI coordinates
atlas = pd.read_csv(f"/data/dy/atlas/upgrade/Schaefer{schaefer}x7_MNI.csv")


In [ ]:
# Export filtered time series for each subject/session and save metadata + ROI info
meta = pd.DataFrame(columns = ['subject_id','condition'])

for sub in tqdm(sub_ls, ncols = 100):
    for i,ses in enumerate(ses_ls):
        # Load BOLD data and select only the chosen RSN ROIs
        bold = pd.read_csv(f"/data/dy/TIS_MDD/BOLD/{sub}/{ses}/{schaefer}.csv").T
        bold = choose_rsn(bold, rsn = chosen_network)
        # Save time series to LEiDA input folder
        sub_id = f"{sub}{i+1}"
        os.makedirs(f"{leida_path}/time_series/{sub_id}",exist_ok = True)
        bold.to_csv(f"{leida_path}/time_series/{sub_id}/{sub_id}.csv",index = False, header = False)
        # Record subject ID and session in metadata
        meta.loc[sub_id,:] = sub_id,ses

meta.to_csv(f"{leida_path}/metadata.csv", index = False)

# Save ROI coordinates and labels for LEiDA
rois_coordinates,rois_labels = chosen_rois(rsn = chosen_network)
rois_coordinates.to_csv(f"{leida_path}/rois_coordinates.csv", index = False)
rois_labels.to_csv(f"{leida_path}/rois_labels.txt", sep = '\t', index = False, header = False)


100%|███████████████████████████████████████████████████████████████| 26/26 [00:03<00:00,  7.63it/s]


In [ ]:
# Import the LEiDA analysis class
from pyleida import Leida


In [ ]:
# Initialize LEiDA with the prepared data path and run the full pipeline
ld = Leida(leida_path)
ld.fit_predict(TR=2,paired_tests=False,n_perm=5_000,save_results=True)


All the data has been sucesfully loaded.

-Creating folder to save results: './LEiDA_results'

-STARTING THE PROCESS:
-Number of subjects: 130

 1) EXTRACTING THE EIGENVECTORS.

SUBJECT ID: sub-0011 (300 volumes)
SUBJECT ID: sub-0012 (300 volumes)
SUBJECT ID: sub-0013 (300 volumes)
SUBJECT ID: sub-0014 (300 volumes)
SUBJECT ID: sub-0015 (300 volumes)
SUBJECT ID: sub-0021 (300 volumes)
SUBJECT ID: sub-0022 (300 volumes)
SUBJECT ID: sub-0023 (300 volumes)
SUBJECT ID: sub-0024 (300 volumes)
SUBJECT ID: sub-0025 (300 volumes)
SUBJECT ID: sub-0051 (300 volumes)
SUBJECT ID: sub-0052 (300 volumes)
SUBJECT ID: sub-0053 (300 volumes)
SUBJECT ID: sub-0054 (300 volumes)
SUBJECT ID: sub-0055 (300 volumes)
SUBJECT ID: sub-0081 (300 volumes)
SUBJECT ID: sub-0082 (300 volumes)
SUBJECT ID: sub-0083 (300 volumes)
SUBJECT ID: sub-0084 (300 volumes)
SUBJECT ID: sub-0085 (300 volumes)
SUBJECT ID: sub-0111 (300 volumes)
SUBJECT ID: sub-0112 (300 volumes)
SUBJECT ID: sub-0113 (300 volumes)
SUBJECT ID: sub-0